In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/uom190346a/water-quality-and-potability/water_potability.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score


In [3]:
df = pd.read_csv('/kaggle/input/datasets/uom190346a/water-quality-and-potability/water_potability.csv')
df


float64_cols = df.select_dtypes(include='float64').columns
df[float64_cols] = df[float64_cols].round(2)

# 2. Export to CSV with float_format='%.2f' so 2 decimal places appear in the file
df.to_csv('water_potability_2dec.csv', index=False, float_format='%.2f')

# 3. Verify raw CSV text and reloaded DataFrame dtypes
with open('water_potability_2dec.csv', 'r') as f:
  for _ in range(5):
    print(f.readline().strip())

df_check = pd.read_csv('water_potability_2dec.csv')
print('\nReloaded dtypes:\n', df_check.dtypes)

df.head()

ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
,204.89,20791.32,7.30,368.52,564.31,10.38,86.99,2.96,0
3.72,129.42,18630.06,6.64,,592.89,15.18,56.33,4.50,0
8.10,224.24,19909.54,9.28,,418.61,16.87,66.42,3.06,0
8.32,214.37,22018.42,8.06,356.89,363.27,18.44,100.34,4.63,0

Reloaded dtypes:
 ph                 float64
Hardness           float64
Solids             float64
Chloramines        float64
Sulfate            float64
Conductivity       float64
Organic_carbon     float64
Trihalomethanes    float64
Turbidity          float64
Potability           int64
dtype: object


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.89,20791.32,7.30,368.52,564.31,10.38,86.99,2.96,0
1,3.72,129.42,18630.06,6.64,NaN,592.89,15.18,56.33,4.50,0
2,8.10,224.24,19909.54,9.28,NaN,418.61,16.87,66.42,3.06,0
3,8.32,214.37,22018.42,8.06,356.89,363.27,18.44,100.34,4.63,0
4,9.09,181.10,17978.99,6.55,310.14,398.41,11.56,32.00,4.08,0


In [4]:
df.head()
df.tail()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
3271,4.67,193.68,47580.99,7.17,359.95,526.42,13.89,66.69,4.44,1
3272,7.81,193.55,17329.80,8.06,NaN,392.45,19.90,NaN,2.80,1
3273,9.42,175.76,33155.58,7.35,NaN,432.04,11.04,69.85,3.30,1
3274,5.13,230.60,11983.87,6.30,NaN,402.88,11.17,77.49,4.71,1
3275,7.87,195.10,17404.18,7.51,NaN,327.46,16.14,78.70,2.31,1


In [5]:
df.isnull().sum()


ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64

In [6]:
df.isnull().sum() / len(df)

ph                 0.149878
Hardness           0.000000
Solids             0.000000
Chloramines        0.000000
Sulfate            0.238400
Conductivity       0.000000
Organic_carbon     0.000000
Trihalomethanes    0.049451
Turbidity          0.000000
Potability         0.000000
dtype: float64

In [7]:

for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

/tmp/ipykernel_16/1533339413.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


In [8]:
df.shape

(3276, 10)

In [9]:

desc_2dec = df.describe().round(2)
print(desc_2dec)

            ph  Hardness    Solids  Chloramines  Sulfate  Conductivity  \
count  3276.00   3276.00   3276.00      3276.00  3276.00       3276.00   
mean      7.07    196.37  22014.09         7.12   333.61        426.21   
std       1.47     32.88   8768.57         1.58    36.14         80.82   
min       0.00     47.43    320.94         0.35   129.00        181.48   
25%       6.28    176.85  15666.69         6.13   317.09        365.74   
50%       7.04    196.96  20927.83         7.13   333.07        421.88   
75%       7.87    216.67  27332.76         8.11   350.38        481.79   
max      14.00    323.12  61227.20        13.13   481.03        753.34   

       Organic_carbon  Trihalomethanes  Turbidity  Potability  
count         3276.00          3276.00    3276.00     3276.00  
mean            14.28            66.41       3.97        0.39  
std              3.31            15.77       0.78        0.49  
min              2.20             0.74       1.45        0.00  
25%          

In [10]:
df.info().round(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3276 entries, 0 to 3275
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ph               3276 non-null   float64
 1   Hardness         3276 non-null   float64
 2   Solids           3276 non-null   float64
 3   Chloramines      3276 non-null   float64
 4   Sulfate          3276 non-null   float64
 5   Conductivity     3276 non-null   float64
 6   Organic_carbon   3276 non-null   float64
 7   Trihalomethanes  3276 non-null   float64
 8   Turbidity        3276 non-null   float64
 9   Potability       3276 non-null   int64  
dtypes: float64(9), int64(1)
memory usage: 256.1 KB


AttributeError: 'NoneType' object has no attribute 'round'

In [ ]:
df.columns

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Potability',hue='Potability', data=df, palette='Set2', legend=False)
plt.title('Distribution of Water Potability')
plt.show()

print(df['Potability'].value_counts(normalize=True) * 100)
    

In [ ]:

df.hist(bins=30, figsize=(12, 10), edgecolor='black')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(df.columns[:-1]): 
    plt.subplot(3, 3, i + 1)
    sns.histplot(data=df, x=col, hue='Potability', kde=True, bins=30, palette='viridis')
    plt.title(f'Distribution of {col}')
    plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

df = pd.read_csv('water_potability_2dec.csv')
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
[sns.kdeplot(df, x=c, hue='Potability', fill=True, ax=ax) for ax, c in zip(axes.flat, df.select_dtypes('float64'))]
plt.tight_layout()

In [ ]:
import numpy as np

# Apply log transformation
df['Solid'] = np.log1p(df["Solids"])
#df = pd.read_csv('water_potability_2dec.csv')
#fig, axes = plt.subplots(3, 3, figsize=(12, 9))
[sns.kdeplot(data=df["Solid"],x="Soilds"]
#plt.tight_layout()

In [ ]:

corr_matrix = df.corr()


plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Water Quality Parameters')
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(data=df.drop(columns=['Potability']), orient='h', palette='Set3')
plt.title('Boxplots for Outlier Detection')
plt.show()

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.drop('Potability')

Q1 = df[numerical_cols].quantile(0.25)
Q3 = df[numerical_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [ ]:
for col in numerical_cols:
    df[col] = np.where(df[col] < lower_bound[col], lower_bound[col], df[col])
    df[col] = np.where(df[col] > upper_bound[col], upper_bound[col], df[col])

In [ ]:
for col in ['ph', 'Sulfate', 'Trihalomethanes']:
    df[col] = df[col].fillna(df.groupby('Potability')[col].transform('median'))

In [ ]:
df.fillna(df.median(), inplace=True)

In [ ]:


X = df.drop(columns=['Potability'])
y = df['Potability']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:

df['Hardness_Solids_Ratio'] = df['Hardness'] / (df['Solids'] + 1e-5)

df['Conductivity_Solids_Ratio'] = df['Conductivity'] / (df['Solids'] + 1e-5)

#
df['Disinfectant_Product'] = df['Chloramines'] * df['Trihalomethanes']

# 4. pH Balance Deviation Indicator
# Pure/ideal drinking water neutral pH is around 7. 
df['pH_Deviation_from_Neutral'] = np.abs(df['ph'] - 7.0)

print("New engineered features added successfully!")
print(df[['Hardness_Solids_Ratio', 'Conductivity_Solids_Ratio', 'Disinfectant_Product', 'pH_Deviation_from_Neutral']].head())

In [ ]:


# ==========================================
# 1. Baseline Model: Logistic Regression
# ==========================================
print("--- Logistic Regression Baseline ---")
# Logistic regression requires scaled data because it relies on distance/coefficients
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, lr_preds))
print("ROC-AUC:", roc_auc_score(y_test, lr_probs))
print("\nClassification Report:\n", classification_report(y_test, lr_preds))


# ==========================================
# 2. Baseline Model: Basic Decision Tree
# ==========================================
print("\n--- Decision Tree Baseline ---")
# Decision trees do not strictly require scaled data, but using X_train/X_test directly is fine
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

dt_preds = dt_model.predict(X_test)
dt_probs = dt_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, dt_preds))
print("ROC-AUC:", roc_auc_score(y_test, dt_probs))
print("\nClassification Report:\n", classification_report(y_test, dt_preds))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# ==========================================
# 1. Random Forest with Class Weight Balancing
# ==========================================
print("--- Random Forest (Balanced) ---")
rf_model = RandomForestClassifier(
    n_estimators=300, 
    max_depth=12, 
    class_weight='balanced',  # Automatically adjusts weights inversely proportional to class frequencies
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

rf_preds = rf_model.predict(X_test_scaled)
rf_probs = rf_model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, rf_preds))
print("ROC-AUC:", roc_auc_score(y_test, rf_probs))
print("\nClassification Report:\n", classification_report(y_test, rf_preds))


# ==========================================
# 2. XGBoost with Scale Pos Weight Balancing
# ==========================================
print("\n--- XGBoost (Balanced) ---")
# Calculate scale_pos_weight for XGBoost: (count of negative class / count of positive class)
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300, 
    learning_rate=0.05, 
    max_depth=6, 
    scale_pos_weight=scale_weight,  # Balances positive vs negative weights in XGBoost
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_scaled, y_train)

xgb_preds = xgb_model.predict(X_test_scaled)
xgb_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, xgb_preds))
print("ROC-AUC:", roc_auc_score(y_test, xgb_probs))
print("\nClassification Report:\n", classification_report(y_test, xgb_preds))

In [ ]:


# Define the grid
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6, 8],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'scale_pos_weight': [scale_weight]
}

xgb_tuned = XGBClassifier(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=xgb_tuned,
    param_distributions=param_dist,
    n_iter=15,
    scoring='roc_auc',
    cv=5,
    verbose=1,
    random_state=42
)

random_search.fit(X_train_scaled, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best CV ROC-AUC:", random_search.best_score_)

# Evaluate best model on test set
best_model = random_search.best_estimator_
best_probs = best_model.predict_proba(X_test_scaled)[:, 1]
print("Test Set ROC-AUC:", roc_auc_score(y_test, best_probs))

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

best_preds = (best_probs >= 0.5).astype(int)
print("Accuracy:", accuracy_score(y_test, best_preds))
print("\nClassification Report:\n", classification_report(y_test, best_preds))

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(y_test, best_probs)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'XGBoost Tuned (ROC-AUC = 0.6738)')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:
import joblib

# Save the best tuned XGBoost model
joblib.dump(best_model, 'water_potability_xgb_model.pkl')

# Save the fitted StandardScaler
joblib.dump(scaler, 'water_potability_scaler.pkl')

print("Model and scaler saved successfully as .pkl files!")

In [ ]:
# 1. Load the saved model and scaler
loaded_model = joblib.load('water_potability_xgb_model.pkl')
loaded_scaler = joblib.load('water_potability_scaler.pkl')

# 2. Prepend/process your new raw input data the exact same way (imputation, scaling, etc.)
# new_data_scaled = loaded_scaler.transform(new_raw_data)

# 3. Predict potability (0 or 1) and probabilities
# predictions = loaded_model.predict(new_data_scaled)
# probabilities = loaded_model.predict_proba(new_data_scaled)[:, 1]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier


# Custom transformer to generate domain features cleanly inside the Pipeline
class WaterDomainFeatures(BaseEstimator, TransformerMixin):

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X_copy = X.copy()
    # 1. Neutrality deviation: distance from pure neutral pH (7.0)
    X_copy["ph_neutral_dev"] = np.abs(X_copy["ph"] - 7.0)

    # 2. Sulfate-to-Solids ratio (add 1e-6 to avoid division by zero)
    X_copy["sulfate_solids_ratio"] = X_copy["Sulfate"] / (
        X_copy["Solids"] + 1e-6
    )

    # 3. Optional high-signal interaction: Chloramines to Organic Carbon (DBP precursor proxy)
    X_copy["chloramine_carbon_ratio"] = X_copy["Chloramines"] / (
        X_copy["Organic_carbon"] + 1e-6
    )
    return X_copy

In [ ]:
# Load your dataset
# df = pd.read_csv("water_potability.csv")
X = df.drop("Potability", axis=1)
y = df["Potability"]

# 1. Stratified split BEFORE any imputation or scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Calculate class imbalance ratio strictly on y_train for XGBoost
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
xgb_scale_pos_weight = neg_count / pos_count

# --- Pipeline A: ExtraTreesClassifier ---
et_pipeline = Pipeline([
    ("features", WaterDomainFeatures()),
    (
        "scaler",
        RobustScaler(),
    ),  # Essential so Solids doesn't skew KNN distances
    ("imputer", KNNImputer(n_neighbors=5, weights="distance")),
    (
        "model",
        ExtraTreesClassifier(
            n_estimators=300,
            max_depth=15,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

# --- Pipeline B: Tuned XGBoost ---
xgb_pipeline = Pipeline([
    ("features", WaterDomainFeatures()),
    ("scaler", RobustScaler()),
    ("imputer", KNNImputer(n_neighbors=5, weights="distance")),
    (
        "model",
        XGBClassifier(
            n_estimators=300,
            max_depth=15,
            learning_rate=0.03,  # Lower LR prevents overfitting at depth=15
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,  # Regularizes deep leaf splits
            scale_pos_weight=xgb_scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

# Fit and evaluate both pipelines
for name, pipe in [("ExtraTrees", et_pipeline), ("XGBoost", xgb_pipeline)]:
  pipe.fit(X_train, y_train)
  preds = pipe.predict(X_test)
  probs = pipe.predict_proba(X_test)[:, 1]

  print(f"=== {name} ===")
  print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")
  print(classification_report(y_test, preds))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier


# Custom transformer to generate domain features cleanly inside the Pipeline
class WaterDomainFeatures(BaseEstimator, TransformerMixin):

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X_copy = X.copy()
    # 1. Neutrality deviation: distance from pure neutral pH (7.0)
    X_copy["ph_neutral_dev"] = np.abs(X_copy["ph"] - 7.0)

    # 2. Sulfate-to-Solids ratio (add 1e-6 to avoid division by zero)
    X_copy["sulfate_solids_ratio"] = X_copy["Sulfate"] / (
        X_copy["Solids"] + 1e-6
    )

    # 3. Optional high-signal interaction: Chloramines to Organic Carbon (DBP precursor proxy)
    X_copy["chloramine_carbon_ratio"] = X_copy["Chloramines"] / (
        X_copy["Organic_carbon"] + 1e-6
    )
    return X_copy

In [ ]:
# Load your dataset
# df = pd.read_csv("water_potability.csv")
X = df.drop("Potability", axis=1)
y = df["Potability"]

# 1. Stratified split BEFORE any imputation or scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Calculate class imbalance ratio strictly on y_train for XGBoost
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
xgb_scale_pos_weight = neg_count / pos_count

# --- Pipeline A: ExtraTreesClassifier ---
et_pipeline = Pipeline([
    ("features", WaterDomainFeatures()),
    (
        "scaler",
        RobustScaler(),
    ),  # Essential so Solids doesn't skew KNN distances
    ("imputer", KNNImputer(n_neighbors=5, weights="distance")),
    (
        "model",
        ExtraTreesClassifier(
            n_estimators=300,
            max_depth=15,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

# --- Pipeline B: Tuned XGBoost ---
xgb_pipeline = Pipeline([
    ("features", WaterDomainFeatures()),
    ("scaler", RobustScaler()),
    ("imputer", KNNImputer(n_neighbors=5, weights="distance")),
    (
        "model",
        XGBClassifier(
            n_estimators=300,
            max_depth=15,
            learning_rate=0.03,  # Lower LR prevents overfitting at depth=15
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,  # Regularizes deep leaf splits
            scale_pos_weight=xgb_scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

# Fit and evaluate both pipelines
for name, pipe in [("ExtraTrees", et_pipeline), ("XGBoost", xgb_pipeline)]:
  pipe.fit(X_train, y_train)
  preds = pipe.predict(X_test)
  probs = pipe.predict_proba(X_test)[:, 1]

  print(f"=== {name} ===")
  print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")
  print(classification_report(y_test, preds))